In [45]:
import osmnx
from osmnx.simplification import consolidate_intersections
from geopandas import GeoDataFrame

In [15]:
from psycopg import connect
from pathlib import Path
import configparser
CONFIG = configparser.ConfigParser()
CONFIG.read(str(Path.home().joinpath('db.cfg'))) #Creates a path to your db.cfg file
dbset = CONFIG['DBSETTINGS']
con = connect(**dbset)

In [56]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from pathlib import Path
import configparser
CONFIG = configparser.ConfigParser()
CONFIG.read(str(Path.home().joinpath('db.cfg'))) #Creates a path to your db.cfg file
connect_url = URL.create("postgresql+psycopg2", **CONFIG['DBSETTINGS'])
engine = create_engine(connect_url)

In [43]:
with con.cursor() as _: 
    here_nodes = GeoDataFrame.from_postgis('''SELECT node_id as osmid, ST_Transform(geom, 2952) geom, 
                                            ST_X(ST_Transform(geom, 2952)) as x, 
                                            st_Y(ST_Transform(geom, 2952)) as y 
                                            FROM congestion.network_nodes_24_4''',
                                           con,
                                           crs=2952,
                                           index_col='osmid')

/data/jupyterhub/.venv/lib/python3.10/site-packages/geopandas/io/sql.py:170: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


In [41]:
with con.cursor() as _:
    here_edges = GeoDataFrame.from_postgis('''SELECT start_vid as u, end_vid as v
                                              , link_dir as key
                                              , ST_Transform(geom, 2952) as geom
                                                    --, geom, length 
                                              FROM congestion.network_links_24_4''',
                                           con,
                                           crs=2952,
                                           index_col=['u','v','key']
                                          )


/data/jupyterhub/.venv/lib/python3.10/site-packages/geopandas/io/sql.py:170: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


In [44]:
here_graph = osmnx.convert.graph_from_gdfs(here_nodes, here_edges)

In [61]:
here_graph_consolidated = consolidate_intersections(here_graph, 
                                                    dead_ends=True,
                                                   reconnect_edges=True,
                                                   tolerance=30)

In [ ]:
here_graph

In [62]:
here_nodes_consolidated = osmnx.convert.graph_to_gdfs(here_graph_consolidated,
                                                      nodes=True,
                                                      edges=False)
here_nodes_consolidated.count()

osmid_original    3265
x                 3265
y                 3265
street_count      3265
geometry          3265
dtype: int64

In [63]:
here_nodes_consolidated.to_postgis('congestion_nodes_osmnx_24_4_30', 
                                       engine,
                                       schema='congestion')